## Bijdrage Raoul Buffels — DeepLearningTeam10

Deze Citi Bike-uitwerking is aangeleverd als de bijdrage van **Raoul Buffels**. Teamleden: Jorre Van Dyck, Milan Wouters, Raoul Buffels en Andrew Noeyens. De concrete menselijke taakverdeling moet het team zelf aanvullen. **AI-gebruik:** Codex hielp met implementatie, uitleg, tests, uitvoering en verpakking.

De codecellen zijn uitgevoerd in de oorspronkelijke lokale `Citybike_full`-omgeving op 9 oktober 2026. Deze export behoudt die echte uitvoer; alleen deze Markdown-cel is toegevoegd. Lokale paden, timings en fingerprints in de rapporten beschrijven de oorspronkelijke uitvoering. Start voor heruitvoering vanuit `SolutionRaoul/Citybike` of de map `notebooks` met de bijbehorende Python 3.11-omgeving. AWS is een afzonderlijke latere taak.

# Random Forest

Bijdragen: door team in te vullen. De training gebeurt offline via `run.py train`. Dit notebook bewaart echte scores en configuraties; geen eindtest wordt voor tuning gebruikt.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Markdown
locations = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'Citybike_full']
ROOT = next((p for p in locations if (p / 'citibike').exists()), None)
if ROOT is None: raise RuntimeError('Start dit notebook vanuit Citybike_full of notebooks.')
sys.path.insert(0, str(ROOT))
from citibike.config import REPORTS, DATA, MODELS
from citibike.datasets import manifest
from citibike.plots import show
def report(name):
    return json.loads((REPORTS / name).read_text(encoding='utf-8'))
def table(name):
    display(pd.read_csv(REPORTS / name, dtype={'start_station_id':'string','end_station_id':'string','station_id':'string'}))


De kern van iedere trainingsstap is hieronder weergegeven. De CLI herhaalt dit voor dezelfde opgeslagen folds, fit preprocessing alleen op de trainingsrijen en bewaart de werkelijke scores. Dit rapportnotebook voert na de eindtest geen nieuwe selectie uit.

```python
fitted = clone(candidate).fit(training[FEATURES], training.rides)
predicted = fitted.predict(validation[FEATURES])
scores = metrics(validation.rides, predicted)
```

In [2]:
from citibike.modeling import candidates
from citibike.features import FEATURES
import joblib
for name in ['random_forest', 'tuned_random_forest']:
    if (MODELS/(name+'.joblib')).exists():
        model=joblib.load(MODELS/(name+'.joblib'))
        display(Markdown('**Opgeslagen estimator: '+name+'**'))
        display(Markdown(str(model.estimator_)))

**Opgeslagen estimator: random_forest**

RandomForestRegressor(max_depth=16, max_features=0.8, min_samples_leaf=8,
                      n_estimators=160, n_jobs=2, random_state=42)

**Opgeslagen estimator: tuned_random_forest**

RandomForestRegressor(max_features=0.6, min_samples_leaf=4, n_estimators=320,
                      n_jobs=2, random_state=42)

Dezelfde chronologische validatie en operationele voorspelfunctie maken de vergelijking eerlijk.

In [3]:
comparison=pd.read_csv(REPORTS/'model_comparison.csv')
display(comparison[comparison.name.isin(['random_forest', 'tuned_random_forest'])])

,name,RMSE,MAE,R2,train_RMSE,train_MAE,train_R2,cv_RMSE,fit_cv_seconds
2,random_forest,1945.709748,1118.646342,0.842674,684.188257,359.153829,0.944902,1460.783347,394.205085
5,tuned_random_forest,1940.492880,1114.479601,0.843517,536.134489,268.116716,0.966168,1433.759353,811.226580


Een lagere trainingsfout kan overfit betekenen. Kies op validatie en bekijk de CV-spreiding en looptijd.

In [4]:
records=report('model_comparison.json')
for record in records:
    if record['name'] in ['random_forest', 'tuned_random_forest']:
        display(Markdown('**'+record['name']+'**'))
        display(pd.DataFrame(record['cv']))
        display(Markdown(str(record['parameters'])))

**random_forest**

,fold,RMSE,MAE,R2,seconds
0,1,1116.667979,710.289700,0.728326,79.827423
1,2,1215.657622,783.153034,0.696952,73.924428
2,3,1116.000881,739.424126,0.590380,69.723780
3,4,1923.862375,1208.712803,0.694658,48.999825
4,5,1931.727881,1146.665688,0.796767,43.112384


{'columns': None, 'estimator__bootstrap': True, 'estimator__ccp_alpha': 0.0, 'estimator__criterion': 'squared_error', 'estimator__max_depth': 16, 'estimator__max_features': 0.8, 'estimator__max_leaf_nodes': None, 'estimator__max_samples': None, 'estimator__min_impurity_decrease': 0.0, 'estimator__min_samples_leaf': 8, 'estimator__min_samples_split': 2, 'estimator__min_weight_fraction_leaf': 0.0, 'estimator__monotonic_cst': None, 'estimator__n_estimators': 160, 'estimator__n_jobs': 2, 'estimator__oob_score': False, 'estimator__random_state': 42, 'estimator__verbose': 0, 'estimator__warm_start': False, 'estimator': 'RandomForestRegressor(max_depth=16, max_features=0.8, min_samples_leaf=8,\n                      n_estimators=160, n_jobs=2, random_state=42)'}

**tuned_random_forest**

,fold,RMSE,MAE,R2,seconds
0,1,1079.045621,692.836563,0.746324,87.175680
1,2,1189.512715,762.030850,0.709847,126.468818
2,3,1101.450573,729.158141,0.600992,138.432619
3,4,1890.356802,1185.626696,0.705201,147.487994
4,5,1908.431056,1123.122751,0.801639,156.848764


{'columns': None, 'estimator__bootstrap': True, 'estimator__ccp_alpha': 0.0, 'estimator__criterion': 'squared_error', 'estimator__max_depth': None, 'estimator__max_features': 0.6, 'estimator__max_leaf_nodes': None, 'estimator__max_samples': None, 'estimator__min_impurity_decrease': 0.0, 'estimator__min_samples_leaf': 4, 'estimator__min_samples_split': 2, 'estimator__min_weight_fraction_leaf': 0.0, 'estimator__monotonic_cst': None, 'estimator__n_estimators': 320, 'estimator__n_jobs': 2, 'estimator__oob_score': False, 'estimator__random_state': 42, 'estimator__verbose': 0, 'estimator__warm_start': False, 'estimator': 'RandomForestRegressor(max_features=0.6, min_samples_leaf=4, n_estimators=320,\n                      n_jobs=2, random_state=42)'}

In [5]:
for path in sorted(REPORTS.glob('*_tuning.csv')):
    if path.stem.split('_tuning')[0] in ['random_forest']:
        display(pd.read_csv(path)[['configuration','cv_RMSE','parameters']])

,configuration,cv_RMSE,parameters
0,1,1485.574606,"{""estimator__n_estimators"": 320, ""estimator__m..."
1,2,1485.152935,"{""estimator__n_estimators"": 240, ""estimator__m..."
2,3,1475.817191,"{""estimator__n_estimators"": 240, ""estimator__m..."
3,4,1433.759353,"{""estimator__n_estimators"": 320, ""estimator__m..."
4,5,1462.581949,"{""estimator__n_estimators"": 320, ""estimator__m..."
5,6,1459.088472,"{""estimator__n_estimators"": 240, ""estimator__m..."
